# Baseline Prediction Fairness Metrics

First map baseline decisions to evaluation applicants by `id_candidat`. Then compare predicted grant rates for All, remote, and center applicants using the qualification proxy, including the Genie/Sante study-bonus alternative. The evaluation set has no grant outcomes, so these are prediction selection rates, not accuracy or measured true fairness.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

candidate_relative_path = Path("equialgo-participants/data/candidats_evaluation.csv")
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / candidate_relative_path).is_file()),
    None,
)
if repository_root is None:
    raise FileNotFoundError(f"Could not find {candidate_relative_path} from {Path.cwd()}")

analysis_dir = repository_root / "Analysis" / "fairness_and_bias_analysis" / "baseline_model_predictions"
predictions_path = analysis_dir / "baseline_predictions.csv"
candidates = pd.read_csv(repository_root / candidate_relative_path)
predictions = pd.read_csv(predictions_path).rename(
    columns={"decision_octroi": "baseline_prediction"}
)

if candidates["id_candidat"].duplicated().any() or predictions["id_candidat"].duplicated().any():
    raise ValueError("Candidate and prediction IDs must be unique for one-to-one mapping.")
mapped = candidates.merge(
    predictions, on="id_candidat", how="outer", validate="one_to_one", indicator=True
)
if not mapped["_merge"].eq("both").all():
    raise ValueError("Prediction IDs do not match the evaluation candidate IDs.")
mapped = mapped.drop(columns="_merge")
print(f"Mapped {len(mapped)} baseline predictions to evaluation applicants.")

remote_regions = {"Bas-Saint-Laurent", "Cote-Nord", "Gaspesie-Iles-de-la-Madeleine"}
mapped["region_group"] = mapped["region_administrative"].apply(
    lambda region: "remote" if region in remote_regions else "center"
)

def minmax_scale(values):
    value_range = values.max() - values.min()
    if value_range == 0:
        return pd.Series(0.5, index=values.index)
    return (values - values.min()) / value_range

r_score = minmax_scale(mapped["cote_r_equivalent"])
income_need_score = 1 - minmax_scale(mapped["revenu_familial_estime"])
hours_burden_score = minmax_scale(mapped["heures_travail_semaine"])

def build_metrics(qualified_column, title, chart_name):
    rows = []
    for group_name in ["All", "remote", "center"]:
        group = mapped if group_name == "All" else mapped[mapped["region_group"] == group_name]
        qualified = group[qualified_column] == 1
        unqualified = ~qualified
        rows.append(
            {
                "group": group_name,
                "% qualified / all": 100 * qualified.mean(),
                "% baseline grants / all": 100 * group["baseline_prediction"].mean(),
                "% baseline grants / qualified": 100 * group.loc[qualified, "baseline_prediction"].mean(),
                "% baseline grants / unqualified": 100 * group.loc[unqualified, "baseline_prediction"].mean(),
            }
        )

    table = pd.DataFrame(rows).set_index("group").round(2)
    print(f"\n{title}")
    print(table.to_string())
    plot_data = table.T[["center", "remote", "All"]].rename(
        index={
            "% qualified / all": "Qualified / all",
            "% baseline grants / all": "Baseline grants / all",
            "% baseline grants / qualified": "Baseline grants / qualified",
            "% baseline grants / unqualified": "Baseline grants / unqualified",
        }
    )
    ax = plot_data.plot(kind="bar", figsize=(11, 6), width=0.78)
    ax.set_title(title)
    ax.set_xlabel("Metric")
    ax.set_ylabel("Applicants (%)")
    ax.set_ylim(0, 100)
    ax.legend(title="Group")
    ax.tick_params(axis="x", rotation=0)
    ax.figure.tight_layout()
    chart_path = analysis_dir / chart_name
    ax.figure.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved chart: {chart_path}")
    return table

mapped["qualified_proxy"] = (
    (3 * r_score + 0.5 * income_need_score + 0.5 * hours_burden_score) / 4
    >= ((3 * r_score + 0.5 * income_need_score + 0.5 * hours_burden_score) / 4).quantile(0.60)
).astype(int)
base_table = build_metrics(
    "qualified_proxy", "Baseline Predictions: Fairness Metrics", "baseline_fairness_metrics.png"
)

study_priority = mapped["programme_etudes"].isin({"Genie", "Sante"}).astype(int)
study_score = (3.5 * r_score + 0.5 * income_need_score + 0.5 * hours_burden_score + 0.5 * study_priority) / 5
mapped["qualified_with_study"] = (study_score >= study_score.quantile(0.60)).astype(int)
study_table = build_metrics(
    "qualified_with_study",
    "Baseline Predictions: Fairness Metrics with Study Bonus",
    "baseline_fairness_metrics_with_study_bonus.png",
)